# Week 2 — Data Cleaning and Preparation

## Internship Project: YouTube Trending Video Analysis — India

The objective of Week 2 was to prepare the raw dataset for further analysis.

The work included checking duplicate records, converting date fields into usable datetime formats, mapping category IDs to category names, and reviewing missing values after cleaning.

In [2]:
import pandas as pd
import json

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/INvideos.csv")

print("Original shape:", df.shape)

Original shape: (37352, 16)


In [3]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 4263


In [4]:
analysis = df.drop_duplicates().copy()

print("Before duplicate removal:", df.shape)
print("After duplicate removal:", analysis.shape)

Before duplicate removal: (37352, 16)
After duplicate removal: (33089, 16)


In [5]:
analysis["trending_date_parsed"] = pd.to_datetime(
    analysis["trending_date"],
    format="%y.%d.%m",
    errors="coerce"
)

analysis["publish_datetime"] = pd.to_datetime(
    analysis["publish_time"],
    errors="coerce",
    utc=True
)

print(
    "Parsed trending dates:",
    analysis["trending_date_parsed"].notna().sum()
)

print(
    "Parsed publish timestamps:",
    analysis["publish_datetime"].notna().sum()
)

Parsed trending dates: 33089
Parsed publish timestamps: 33089


In [6]:
with open("../data/IN_category_id.json", encoding="utf-8") as f:
    categories = json.load(f)

cat_map = {
    int(x["id"]): x["snippet"]["title"]
    for x in categories["items"]
}

analysis["category_name"] = (
    analysis["category_id"]
    .map(cat_map)
)

In [7]:
unmapped = analysis.loc[
    analysis["category_name"].isna(),
    "category_id"
].unique()

print("Unmapped category IDs:", unmapped)

Unmapped category IDs: [29]


In [8]:
missing_after_cleaning = (
    analysis.isna()
    .sum()
    .sort_values(ascending=False)
)

missing_after_cleaning[missing_after_cleaning > 0]

description      527
category_name    103
dtype: int64

In [9]:
analysis.dtypes

video_id                               object
trending_date                          object
title                                  object
channel_title                          object
category_id                             int64
publish_time                           object
tags                                   object
views                                   int64
likes                                   int64
dislikes                                int64
comment_count                           int64
thumbnail_link                         object
comments_disabled                        bool
ratings_disabled                         bool
video_error_or_removed                   bool
description                            object
trending_date_parsed           datetime64[ns]
publish_datetime          datetime64[ns, UTC]
category_name                          object
dtype: object

In [11]:
print("Rows:", len(analysis))
print("Columns:", len(analysis.columns))
print("Duplicate rows remaining:", analysis.duplicated().sum())
print(
    "Invalid trending dates:",
    analysis["trending_date_parsed"].isna().sum()
)
print(
    "Invalid publish timestamps:",
    analysis["publish_datetime"].isna().sum()
)

Rows: 33089
Columns: 19
Duplicate rows remaining: 0
Invalid trending dates: 0
Invalid publish timestamps: 0


## Week 2 Observations

The raw dataset was converted into a cleaner analysis-ready copy.

Exact duplicate rows were removed, date fields were converted into datetime formats, and category IDs were mapped using the provided category metadata.

Missing descriptions were retained because a missing description does not represent a numerical zero.

The cleaned dataset will be used for further transformation and exploratory analysis in the following weeks.